# Iteración del Mapa de Autocorrelación Espacial
Este notebook está preparado para probar y ajustar los detalles visuales del Índice de Moran sobre el mapa de Chile.

In [ ]:
import pandas as pd
import geopandas as gpd
from libpysal.weights import KNN
import esda
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Cargar datos
import pandas as pd
import geopandas as gpd
from shapely.geometry import MultiPolygon

df_full = pd.read_parquet('../Data/casen_2024.parquet')
try:
    gdf_regiones = gpd.read_file('../Data/regiones.geojson', engine='pyogrio')
except Exception:
    gdf_regiones = gpd.read_file('../Data/regiones.geojson')

# Filtrar Isla de Pascua y Juan Fernández (geometrías al oeste de -76° de longitud)
new_geoms = []
for geom in gdf_regiones.geometry:
    if geom is None:
        new_geoms.append(None)
        continue
    if geom.geom_type == 'MultiPolygon':
        clean_polys = [p for p in geom.geoms if p.bounds[0] >= -76]
        new_geoms.append(MultiPolygon(clean_polys) if clean_polys else None)
    elif geom.geom_type == 'Polygon':
        new_geoms.append(geom if geom.bounds[0] >= -76 else None)
    else:
        new_geoms.append(geom)

gdf_regiones = gdf_regiones.copy()
gdf_regiones.geometry = new_geoms
gdf_regiones = gdf_regiones[gdf_regiones.geometry.notnull()]


In [ ]:
# Procesamiento y cálculo de Moran's I
# 1. Calculamos el ingreso promedio por región
df_r = df_full.groupby('region')['yoprcor'].mean().reset_index()

# 2. Hacemos el join con las geometrías
gdf = gdf_regiones.merge(df_r, left_on='codregion', right_on='region', how='inner')

# 3. Definimos los pesos espaciales (KNN=2 para evitar islas desconectadas en Chile)
w = KNN.from_dataframe(gdf, k=2)
w.transform = 'r'

# 4. Calculamos Moran's I
y = gdf['yoprcor'].values
moran = esda.Moran(y, w)

print(f"Índice de Moran (I): {moran.I:.3f}")
print(f"P-value: {moran.p_sim:.3f}")


In [ ]:
# Visualización del Mapa Principal (Coropleta dividido en 2)
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.ticker as ticker

fig, (ax_n, ax_s) = plt.subplots(1, 2, figsize=(8, 7))
vmin = gdf['yoprcor'].min()
vmax = gdf['yoprcor'].max()

north_regions = [15, 1, 2, 3, 4, 5, 13, 6, 7, 16, 8]
south_regions = [9, 14, 10, 11, 12]

gdf_north = gdf[gdf['codregion'].isin(north_regions)]
gdf_south = gdf[gdf['codregion'].isin(south_regions)]

# Graficar Norte y Centro
gdf_north.plot(
    column='yoprcor', 
    cmap='YlOrRd', 
    linewidth=0.2, 
    ax=ax_n, 
    edgecolor='gray', 
    vmin=vmin, 
    vmax=vmax
)
ax_n.axis('off')
ax_n.set_title("Zona Norte y Centro", fontsize=10)

# Graficar Sur
gdf_south.plot(
    column='yoprcor', 
    cmap='YlOrRd', 
    linewidth=0.2, 
    ax=ax_s, 
    edgecolor='gray', 
    vmin=vmin, 
    vmax=vmax
)
ax_s.axis('off')
ax_s.set_title("Zona Sur", fontsize=10)

# Añadir etiquetas con los números de las regiones
region_labels = {
    15: "XV", 1: "I", 2: "II", 3: "III", 4: "IV", 5: "V", 13: "RM", 
    6: "VI", 7: "VII", 16: "XVI", 8: "VIII", 9: "IX", 14: "XIV", 
    10: "X", 11: "XI", 12: "XII"
}

# Anotaciones Zona Norte y Centro
for idx, row in gdf_north.iterrows():
    rep = row['geometry'].representative_point()
    label = region_labels.get(row['codregion'], str(row['codregion']))
    
    xytext = (5, 0)
    ha = 'left'
    va = 'center'
    
    if row['codregion'] == 13: # RM
        xytext = (12, -4)
    elif row['codregion'] == 5: # V
        xytext = (-12, 5)
        ha = 'right'
    elif row['codregion'] == 6: # VI
        xytext = (-12, -5)
        ha = 'right'
    elif row['codregion'] == 15: # XV
        xytext = (8, 0)
    elif row['codregion'] == 1: # I
        xytext = (8, 0)
    elif row['codregion'] == 2: # II
        xytext = (8, 0)
        
    ax_n.annotate(
        label, 
        xy=(rep.x, rep.y), 
        xytext=xytext, 
        textcoords="offset points", 
        fontsize=7.5, 
        fontweight='bold',
        color='#333333',
        ha=ha,
        va=va,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", lw=0.3, alpha=0.85)
    )

# Anotaciones Zona Sur
for idx, row in gdf_south.iterrows():
    rep = row['geometry'].representative_point()
    label = region_labels.get(row['codregion'], str(row['codregion']))
    
    xytext = (10, 0)
    ha = 'left'
    va = 'center'
    
    if row['codregion'] == 11: # XI
        xytext = (15, 0)
    elif row['codregion'] == 12: # XII
        xytext = (15, 10)
        
    ax_s.annotate(
        label, 
        xy=(rep.x, rep.y), 
        xytext=xytext, 
        textcoords="offset points", 
        fontsize=7.5, 
        fontweight='bold',
        color='#333333',
        ha=ha,
        va=va,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", lw=0.3, alpha=0.85)
    )

# Leyenda/barra de color compartida más pequeña
sm = cm.ScalarMappable(cmap='YlOrRd', norm=colors.Normalize(vmin=vmin, vmax=vmax))
sm._A = []
cbar = fig.colorbar(sm, ax=[ax_n, ax_s], orientation='horizontal', shrink=0.5, pad=0.05)
cbar.set_label('Ingreso Promedio (CLP)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Formatear etiquetas de la barra en formato $K
formatter = ticker.FuncFormatter(lambda x, pos: f'${x/1000:,.0f}K'.replace(',', '.'))
cbar.ax.xaxis.set_major_formatter(formatter)

fig.suptitle(f"Ingreso Promedio por Región\nÍndice de Moran: {moran.I:.3f} (p-val: {moran.p_sim:.3f})", fontsize=12, y=0.98)
plt.show()


In [ ]:
# Opcional: Gráfico de Dispersión de Moran (Moran Scatterplot)
from splot.esda import plot_moran

fig, ax = plt.subplots(figsize=(7, 7))
plot_moran(moran, zstandard=True, ax=ax)
plt.show()
